In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import csv
import os
import time
import re

# --- ISRE STEP 1: CONFIG ---
START_URL = "https://pubsonline.informs.org/journal/isre"
START_YEAR = 2010   # scrape from 2010
END_YEAR = 2025

ISRE_ISSUES_CSV = os.path.join(os.getcwd(), "ISRE_Issues.csv")
print(f"ISRE issues CSV will be saved to: {ISRE_ISSUES_CSV}")
print(f"Current working directory: {os.getcwd()}\n")

driver = webdriver.Chrome()


def write_isre_csv(rows):
    file_exists = os.path.exists(ISRE_ISSUES_CSV) and os.path.getsize(ISRE_ISSUES_CSV) > 0
    with open(ISRE_ISSUES_CSV, mode="a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if not file_exists:
            w.writerow(["Title", "URL", "Volume Issue", "Vol Issue Year"])
            print(f"Created CSV file: {ISRE_ISSUES_CSV}")
        w.writerows(rows)
    print(f"  ✓ Saved {len(rows)} articles to {ISRE_ISSUES_CSV}")


def scrape_isre_issue(issue_url, vol_issue, year):
    """Scrape all article full-text URLs from a single ISRE issue page."""
    try:
        print(f"\nScraping issue: {vol_issue} ({year})")
        print(f"URL: {issue_url}")
        driver.get(issue_url)
        driver.implicitly_wait(15)
        time.sleep(2)

        print(f"  Page loaded: {driver.title}")

        # Full-text links usually look like /doi/full/10.xxxx/...
        links = driver.find_elements(By.CSS_SELECTOR, "a[href*='/doi/full/']")
        print(f"  Found {len(links)} potential full-text links")

        seen = set()
        rows = []
        for a in links:
            try:
                href = a.get_attribute("href") or ""
                if not href.startswith("http"):
                    continue
                if href in seen:
                    continue
                seen.add(href)

                # Try to get a reasonable title from nearby elements; fall back to N/A
                title = a.text.strip()
                if not title or len(title) < 5:
                    try:
                        container = a.find_element(
                            By.XPATH,
                            "ancestor::*[contains(@class,'article') or contains(@class,'issue-item')][1]"
                        )
                        try:
                            t_el = container.find_element(By.CSS_SELECTOR, "span.hlFld-Title, h5, h4")
                            title = t_el.text.strip()
                        except:
                            pass
                    except:
                        pass
                if not title:
                    title = "N/A"

                rows.append([title, href, vol_issue, str(year)])
                print(f"    ✓ {title[:70]}...")
            except Exception as e:
                print(f"    Error reading article link: {e}")
                continue

        if rows:
            write_isre_csv(rows)
            return len(rows)
        else:
            print("  ⚠ No article rows found on this issue page")
            return 0

    except Exception as e:
        print(f"  ✗ Error scraping ISRE issue page: {e}")
        return 0


def scrape_isre_year(year_url, year):
    """Scrape all issues for a given ISRE year page."""
    try:
        print("\n" + "=" * 60)
        print(f"Scraping ISRE year {year}: {year_url}")
        print("=" * 60)

        driver.get(year_url)
        driver.implicitly_wait(15)
        time.sleep(3)

        print(f"Page title: {driver.title}")

        issue_links = []
        # On year pages, issues are typically linked as /toc/isre/VOLUME/ISSUE
        candidates = driver.find_elements(By.CSS_SELECTOR, "a[href*='/toc/isre/']")
        print(f"  Found {len(candidates)} candidate issue links")

        for l in candidates:
            try:
                href = l.get_attribute("href") or ""
                text = (l.text or "").strip()
                if "/toc/isre/" in href and href.startswith("http"):
                    issue_links.append((href, text))
            except:
                continue

        # Deduplicate by URL
        unique = {}
        for href, text in issue_links:
            if href not in unique:
                unique[href] = text or href

        print(f"  Total unique issues: {len(unique)}")

        total_articles = 0
        for href, text in unique.items():
            vol_issue = text if text else f"ISRE issue ({year})"
            total_articles += scrape_isre_issue(href, vol_issue, year)
            time.sleep(1)

        print(f"Year {year}: scraped {total_articles} articles")
        return total_articles

    except Exception as e:
        print(f"Error scraping ISRE year page {year_url}: {e}")
        return 0


# --- Main ISRE Step 1 logic ---
try:
    driver.get(START_URL)
    driver.implicitly_wait(15)
    time.sleep(2)

    print("Starting ISRE scraper (step 1: collect article URLs)...")
    print(f"Journal home: {START_URL}\n")

    # Build year URLs directly using the LOI "group" pattern.
    # 2010-2019 live under d2010, 2020-2025 under d2020.
    year_links = {}
    for year in range(START_YEAR, END_YEAR + 1):
        if year <= 2019:
            group_base = "d2010"
        else:
            group_base = "d2020"
        href = f"https://pubsonline.informs.org/loi/isre/group/{group_base}.y{year}"
        year_links[year] = href
        print(f"  ✓ Year {year} URL: {href}")

    print("\nFound ISRE year links:")
    for y in sorted(year_links.keys()):
        print(f"  {y}: {year_links[y]}")

    total = 0
    for y in sorted(year_links.keys()):
        total += scrape_isre_year(year_links[y], y)
        time.sleep(2)

    print("\n" + "=" * 60)
    print("ISRE step 1 complete")
    print(f"Total ISRE article URLs scraped: {total}")
    print("=" * 60)

finally:
    driver.quit()

In [ ]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
from datetime import datetime
import csv
import os
import time
import random
import re

# --- ISRE STEP 2: CONFIG ---
START_INDEX = 0    # adjust range as needed
END_INDEX = 200    # e.g., run in batches
WAIT_SEC = 15

ISRE_CSV_PATH = os.path.join(os.getcwd(), "ISRE_Issues.csv")
ISRE_OUT_FILE = os.path.join(os.getcwd(), "ISRE_article_data.csv")
ISRE_JOURNAL_TITLE = "Information Systems Research"


def clean_text(s):
    if not s:
        return "N/A"
    s = re.sub(r"\s+", " ", str(s).strip())
    return s if s else "N/A"


def parse_month_year(date_str):
    if not date_str:
        return "N/A"
    date_str = date_str.strip()
    for fmt in ("%Y/%m/%d", "%Y-%m-%d", "%b %d, %Y", "%B %d, %Y"):
        try:
            d = datetime.strptime(date_str, fmt)
            return d.strftime("%B %Y")
        except Exception:
            pass
    return clean_text(date_str)


def get_driver():
    opts = Options()
    # Let images load (helps if there are any verification prompts)
    prefs = {"profile.managed_default_content_settings.images": 1}
    opts.add_experimental_option("prefs", prefs)
    return webdriver.Chrome(options=opts)


def extract_article_details(html):
    soup = BeautifulSoup(html, "html.parser")
    meta = {}
    for m in soup.find_all("meta"):
        n = m.get("name")
        c = m.get("content")
        if n and c:
            meta[n.strip()] = c.strip()

    title = clean_text(meta.get("citation_title"))
    if title == "N/A":
        h1 = soup.select_one("h1")
        title = clean_text(h1.get_text()) if h1 else "N/A"

    abstract = "N/A"
    if meta.get("citation_abstract"):
        abstract = clean_text(meta.get("citation_abstract"))
    else:
        abs_el = soup.select_one(".abstract, .abstractSection, .article-abstract, #abstract")
        if abs_el:
            abstract = clean_text(abs_el.get_text())

    kw = []
    for k in soup.find_all("meta", attrs={"name": "citation_keywords"}):
        if k.get("content"):
            kw.append(clean_text(k.get("content")))
    keywords = clean_text(str(sorted(set([x for x in kw if x and x != "N/A"]))))

    vol = clean_text(meta.get("citation_volume"))
    issue = clean_text(meta.get("citation_issue"))
    volume_issue = "N/A"
    if vol != "N/A" and issue != "N/A":
        volume_issue = f"Volume {vol}, Issue {issue}"
    elif vol != "N/A":
        volume_issue = f"Volume {vol}"
    elif issue != "N/A":
        volume_issue = f"Issue {issue}"

    pub_date = meta.get("citation_publication_date") or meta.get("citation_date")
    month_year = parse_month_year(pub_date)

    return title, abstract, keywords, volume_issue, month_year


def extract_isre_authors(driver):
    """Extract authors, emails, and affiliations for ISRE articles.
    Uses citation_* meta tags where available; falls back to simple heuristics.
    """

    def _extract_email(text):
        if not text:
            return "N/A"
        m = re.search(r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}", text, flags=re.I)
        return clean_text(m.group(0)) if m else "N/A"

    def _dedupe(items):
        uniq, seen = [], set()
        for a in items:
            key = (a.get("name"), a.get("email"), a.get("address"))
            if key in seen:
                continue
            seen.add(key)
            uniq.append(a)
        return uniq

    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")

    # Preferred: citation_* meta tags provided by the publisher
    meta_authors = [m.get("content", "").strip() for m in soup.select('meta[name="citation_author"]') if m.get("content")]
    meta_insts = [m.get("content", "").strip() for m in soup.select('meta[name="citation_author_institution"]') if m.get("content")]
    meta_emails = [m.get("content", "").strip() for m in soup.select('meta[name="citation_author_email"]') if m.get("content")]

    if meta_authors and (meta_insts or meta_emails):
        out = []
        for i, name in enumerate(meta_authors):
            inst = meta_insts[i] if i < len(meta_insts) else "N/A"
            em = meta_emails[i] if i < len(meta_emails) else "N/A"
            out.append({
                "name": clean_text(name),
                "email": clean_text(em),
                "address": clean_text(inst),
            })
        return _dedupe([a for a in out if a["name"] != "N/A"])

    # Fallback: simple author list without tooltips/hover (ISRE typically exposes metadata)
    authors = []
    for a in soup.select("a[rel='author'], .entryAuthor a"):
        name = clean_text(a.get_text())
        if name == "N/A":
            continue
        email = "N/A"
        href = a.get("href", "")
        if href.startswith("mailto:"):
            email = clean_text(href.replace("mailto:", "").split("?")[0])
        authors.append({"name": name, "email": email, "address": "N/A"})

    return _dedupe(authors)


if not os.path.exists(ISRE_CSV_PATH):
    print(f"Error: {ISRE_CSV_PATH} not found. Run ISRE step 1 first.")
    raise SystemExit


df_isre = pd.read_csv(ISRE_CSV_PATH)

if not os.path.exists(ISRE_OUT_FILE):
    with open(ISRE_OUT_FILE, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow([
            "URL",
            "Journal_Title",
            "Article_Title",
            "Volume_Issue",
            "Month_Year",
            "Abstract",
            "Keywords",
            "Author_name",
            "Author_email",
            "Author_Address",
        ])


driver = get_driver()
wait = WebDriverWait(driver, WAIT_SEC)

def _looks_like_cloudflare_challenge(d):
    try:
        u = (d.current_url or "").lower()
        t = (d.title or "").lower()
        if "/cdn-cgi/" in u:
            return True
        if "just a moment" in t:
            return True
        src = (d.page_source or "").lower()
        if "cf-turnstile" in src or "challenge-platform" in src or "cf_chl" in src:
            return True
    except Exception:
        return False
    return False

try:
    max_idx = min(END_INDEX, len(df_isre))
    for i in range(START_INDEX, max_idx):
        row = df_isre.iloc[i]
        url = str(row.get("URL", "")).strip()
        if not url:
            continue

        print(f"[{i+1}/{max_idx}] {url}")
        driver.get(url)

        time.sleep(random.uniform(8, 15))

        if _looks_like_cloudflare_challenge(driver):
            print("Verification detected. Solve it in the browser; the script will continue once the article loads.")
            try:
                WebDriverWait(driver, 300).until(lambda d: not _looks_like_cloudflare_challenge(d))
            except Exception:
                pass

        # Wait for the main article title to appear
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))

        html = driver.page_source
        article_title, abstract, keywords, volume_issue, month_year = extract_article_details(html)

        authors = extract_isre_authors(driver)
        if not authors:
            authors = [{"name": "N/A", "email": "N/A", "address": "N/A"}]

        with open(ISRE_OUT_FILE, "a", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            for a in authors:
                w.writerow([
                    url,
                    ISRE_JOURNAL_TITLE,
                    article_title,
                    volume_issue,
                    month_year,
                    abstract,
                    keywords,
                    a["name"],
                    a["email"],
                    a["address"],
                ])

        print(f"   Saved: {len(authors)} author rows")

finally:
    driver.quit()
    print(f"\nDone. ISRE output saved to: {ISRE_OUT_FILE}")